# Venue Tileset Export

Converts one of the isochrone GeoJSON outputs (e.g. `isochrones_weekday_peak.geojson`)
into a **separate PMTiles tileset per `venue_id`**, using `tippecanoe`.

**Prerequisite:** `tippecanoe` must be installed and on your `PATH`
(e.g. `brew install tippecanoe` on macOS). This notebook does not install it.

Each venue's features are written to a temporary GeoJSON, tiled with
tippecanoe, and the temporary file is deleted afterward — mirroring the
single-file approach you're already using elsewhere, just looped per venue.

In [8]:
import json
import subprocess
import shutil
from pathlib import Path

import geopandas as gpd
from shapely.validation import make_valid
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon


## Configuration

Point `INPUT_FILE` at whichever of the six output GeoJSONs you want to tile
(`isochrones_weekday_peak.geojson`, `isochrones_peak_all.geojson`, etc.) —
run this notebook once per file if you want tilesets for more than one.

In [ ]:
INPUT_FILE = "../../data/mobility/isochrones_peak_all.geojson"   # change to whichever output file you want to tile
OUTPUT_DIR = Path("../../data/mobility/commute_time_peak")        # final .pmtiles files land here, one per venue_id, change folder for peak vs off-peak, etc., drag to static folder for use

LAKE_FILE = "../../data/geo/lake-ontario.geojson"     # subtracted from input geometries before tiling
TEMP_DIR = Path("_tileset_tmp")      # scratch space for per-venue GeoJSON, cleaned up after

MIN_ZOOM = 0
MAX_ZOOM = 14

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)


## Subtract Lake Ontario

Subtracts `lake-ontario.geojson` from every input geometry before any
tileset generation, so isochrone rings that geometrically extend into
the lake (since the transit network obviously can't be reached by
walking/transit there) have that portion removed.


In [10]:
def _to_polygonal(geom):
    """make_valid() can return a GeometryCollection with stray points/lines
    mixed in alongside the polygon(s) we actually want. Keep only the
    polygonal parts."""
    if isinstance(geom, (Polygon, MultiPolygon)):
        return geom
    if isinstance(geom, GeometryCollection):
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if not polys:
            return Polygon()  # empty
        return MultiPolygon(polys) if len(polys) > 1 else polys[0]
    return Polygon()  # empty — unexpected geometry type


lake = gpd.read_file(LAKE_FILE)
input_gdf = gpd.read_file(INPUT_FILE)

if input_gdf.crs != lake.crs:
    lake = lake.to_crs(input_gdf.crs)

# Repair invalid geometries on both sides before any boolean operation —
# GEOS raises TopologyException on self-intersecting polygons (a known
# artifact of the raster-vectorize step upstream) otherwise.
input_gdf["geometry"] = input_gdf["geometry"].apply(
    lambda g: g if g.is_valid else _to_polygonal(make_valid(g))
)
lake["geometry"] = lake["geometry"].apply(
    lambda g: g if g.is_valid else _to_polygonal(make_valid(g))
)

# Merge every polygon in the lake file into one geometry, then subtract
# it from EVERY feature in the input file directly.
lake_union = lake.geometry.unary_union

input_gdf["geometry"] = input_gdf["geometry"].apply(
    lambda g: g.difference(lake_union)
)

before_count = len(input_gdf)

# Drop any feature that no longer has area after subtraction (i.e. was
# entirely within the lake to begin with).
subtracted_gdf = input_gdf[~input_gdf.geometry.is_empty].copy()

print(f"Subtracted lake from {before_count} input features -> {len(subtracted_gdf)} features with remaining area")
print(f"Lake extent: {lake.total_bounds}")
print(f"Input extent before subtraction: {input_gdf.total_bounds}")
print(f"Extent after subtraction: {subtracted_gdf.total_bounds}")

# Convert back to plain GeoJSON feature dicts so the rest of the notebook
# (grouping by venue_id, tippecanoe export) works unchanged.
data = json.loads(subtracted_gdf.to_json())


/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_67979/3291665481.py:33: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  lake_union = lake.geometry.unary_union


Subtracted lake from 330 input features -> 330 features with remaining area
Lake extent: [-79.8906559  43.1805016 -78.9381145  43.8241762]
Input extent before subtraction: [-79.76674123  43.50643141 -79.03484711  43.9088509 ]
Extent after subtraction: [-79.76674123  43.50643141 -79.03484711  43.9088509 ]


## Group clipped features by `venue_id`

Each feature in the (now-clipped) input is one donut ring for one venue at
one cutoff band. This groups them so every venue's full set of rings ends
up in the same tileset.

In [11]:
features_by_venue = {}
for feature in data["features"]:
    venue_id = feature.get("properties", {}).get("venue_id")
    if venue_id is None:
        continue
    features_by_venue.setdefault(venue_id, []).append(feature)

print(f"Grouped {len(data['features'])} features across {len(features_by_venue)} venues")


Grouped 330 features across 83 venues


## Per-venue tileset export

Writes one venue's features to a temporary GeoJSON, runs `tippecanoe` on
just that file, then deletes the temporary GeoJSON — same pattern as the
single-file tileset export, just scoped to one venue at a time.

In [12]:
def export_tileset_for_venue(venue_id, features, output_dir, temp_dir, min_zoom, max_zoom):
    temp_geojson = temp_dir / f"venue_{venue_id}.geojson"
    pmtiles_file = output_dir / f"venue_{venue_id}.pmtiles"

    feature_collection = {"type": "FeatureCollection", "features": features}
    with open(temp_geojson, "w") as f:
        json.dump(feature_collection, f)

    subprocess.run([
        "tippecanoe",
        f"--minimum-zoom={min_zoom}",
        f"--maximum-zoom={max_zoom}",
        "-o", str(pmtiles_file),
        "--no-tile-size-limit",
        "--no-feature-limit",
        str(temp_geojson),
        "--force",
    ], check=True)

    temp_geojson.unlink()

    return pmtiles_file


## Run for every venue

Builds one `.pmtiles` file per `venue_id` into `OUTPUT_DIR`. Prints
progress as it goes, since tippecanoe runs sequentially and this can take
a while for a large venue list.

In [13]:
results = []

for i, (venue_id, features) in enumerate(features_by_venue.items(), start=1):
    print(f"[{i}/{len(features_by_venue)}] Building tileset for venue_id={venue_id} "
          f"({len(features)} features)...")
    pmtiles_file = export_tileset_for_venue(
        venue_id, features, OUTPUT_DIR, TEMP_DIR, MIN_ZOOM, MAX_ZOOM
    )
    results.append(pmtiles_file)

print(f"\nDone. {len(results)} tilesets written to {OUTPUT_DIR}/")


[1/83] Building tileset for venue_id=1 (4 features)...


For layer 0, using name "venue_1"
4 features, 49250 bytes of geometry and attributes, 139 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  14/4570/5979  
For layer 0, using name "venue_10"
4 features, 61149 bytes of geometry and attributes, 140 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes


[2/83] Building tileset for venue_id=10 (4 features)...


  99.9%  14/4584/5976  
For layer 0, using name "venue_11"


[3/83] Building tileset for venue_id=11 (4 features)...


...

For layer 0, using name "venue_82"
4 features, 48813 bytes of geometry and attributes, 140 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  14/4574/5971  
For layer 0, using name "venue_83"


[82/83] Building tileset for venue_id=83 (4 features)...


4 features, 57260 bytes of geometry and attributes, 140 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes


[83/83] Building tileset for venue_id=9 (2 features)...

Done. 83 tilesets written to data/commute_time_peak/


  99.9%  14/4578/5976  
For layer 0, using name "venue_9"
2 features, 2997 bytes of geometry and attributes, 119 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  97.0%  14/4579/5981  


## Cleanup

Removes the temporary per-venue GeoJSON scratch directory. Safe to run
even if some tileset builds failed partway through — only touches
`TEMP_DIR`, not `OUTPUT_DIR`.

In [14]:
shutil.rmtree(TEMP_DIR, ignore_errors=True)
print("Cleaned up temp directory.")


Cleaned up temp directory.
